In [3]:
import os
import shutil
from sklearn.model_selection import train_test_split

# SPLIT FOR CONCRETE_CRACK DATASET

- Classification model
- Has Positive and Negative labeling

In [8]:
# Change these paths and parameters as needed
source_dir = 'data/raw/concrete_crack'
dest_dir = 'data/processed/concrete_crack_75_10_15/'
train_size = 0.75
val_size = 0.10
test_size = 0.15

classes = ['Positive', 'Negative']

for cls in classes:
    cls_path = os.path.join(source_dir, cls)
    images = os.listdir(cls_path)

    # First split: train vs temp (val+test)
    train_imgs, temp_imgs = train_test_split(
        images, test_size=(1 - train_size), random_state=42
    )

    # Second split: val vs test
    val_ratio = val_size / (val_size + test_size)
    val_imgs, test_imgs = train_test_split(
        temp_imgs, test_size=(1 - val_ratio), random_state=42
    )

    splits = {
        'train': train_imgs,
        'val': val_imgs,
        'test': test_imgs
    }

    # Copy files
    for split_name, split_files in splits.items():
        split_dir = os.path.join(dest_dir, split_name, cls)
        os.makedirs(split_dir, exist_ok=True)

        for file in split_files:
            src = os.path.join(cls_path, file)
            dst = os.path.join(split_dir, file)
            shutil.copy(src, dst)

print("Dataset split complete.")

Dataset split complete.


# SPLIT FOR CRACKFOREST DATASET
- Segmentation model
- Has Images and Masks

In [6]:
source_dir = 'data/raw/crackforest'
dest_dir = 'data/processed/crackforest_75_10_15/'
train_size = 0.75
val_size = 0.10
test_size = 0.15

# Subfolders
images_dir = os.path.join(source_dir, 'Images')
masks_dir = os.path.join(source_dir, 'Masks')

# ---- DEBUG CHECKS ----
print("Working directory:", os.getcwd())
print("Source exists:", os.path.exists(source_dir))
print("Images dir:", images_dir, "| Exists:", os.path.exists(images_dir))
print("Masks dir:", masks_dir, "| Exists:", os.path.exists(masks_dir))

if not os.path.exists(images_dir):
    raise FileNotFoundError(f"Images folder not found: {images_dir}")

if not os.path.exists(masks_dir):
    raise FileNotFoundError(f"Masks folder not found: {masks_dir}")

# ---- LOAD FILES ----
# CrackForest often uses .bmp → include it
valid_exts = ('.jpg', '.png', '.bmp')

images = [
    f for f in os.listdir(images_dir)
    if f.lower().endswith(valid_exts)
]

print(f"Total images found: {len(images)}")

if len(images) == 0:
    raise ValueError("No images found. Check folder path or file extensions.")

# ---- VERIFY MASK PAIRS ----
'''
missing_masks = [
    f for f in images
    if not os.path.exists(os.path.join(masks_dir, f))
]

if missing_masks:
    print("Warning: Some masks are missing!")
    print("Example missing:", missing_masks[:5])
    raise ValueError("Mismatch between Images and Masks filenames.")
'''

# ---- SPLITTING ----
train_imgs, temp_imgs = train_test_split(
    images, test_size=(1 - train_size), random_state=42
)

val_ratio = val_size / (val_size + test_size)

val_imgs, test_imgs = train_test_split(
    temp_imgs, test_size=(1 - val_ratio), random_state=42
)

splits = {
    'train': train_imgs,
    'val': val_imgs,
    'test': test_imgs
}

# ---- COPY FILES ----
for split_name, split_files in splits.items():
    img_split_dir = os.path.join(dest_dir, split_name, 'Images')
    mask_split_dir = os.path.join(dest_dir, split_name, 'Masks')

    os.makedirs(img_split_dir, exist_ok=True)
    os.makedirs(mask_split_dir, exist_ok=True)

    print(f"\nProcessing {split_name} ({len(split_files)} files)...")

    for file in split_files:
        base_name = os.path.splitext(file)[0]

        img_src = os.path.join(images_dir, file)
        mask_name = base_name + "_label.PNG"
        mask_src = os.path.join(masks_dir, mask_name)

        if not os.path.exists(mask_src):
            print(f"Missing mask for {file} → expected {mask_name}")
            continue

        img_dst = os.path.join(img_split_dir, file)
        mask_dst = os.path.join(mask_split_dir, mask_name)

        shutil.copy(img_src, img_dst)
        shutil.copy(mask_src, mask_dst)

print("\n✅ CrackForest dataset split complete.")

Working directory: c:\Users\hiend\OneDrive\Documentos\Coding\DCSI_4850\Concrete-Crack-Detection
Source exists: True
Images dir: data/raw/crackforest\Images | Exists: True
Masks dir: data/raw/crackforest\Masks | Exists: True
Total images found: 118

Processing train (88 files)...

Processing val (12 files)...

Processing test (18 files)...

✅ CrackForest dataset split complete.


# SPLIT FOR SDNET18

- Classification model
- Has Decks/ Pavements/ Walls/ with Cracked/ and Non-cracked

In [9]:
source_dir = 'data/raw/structural_defects'
dest_dir = 'data/processed/structural_defects_75_10_15/'
train_size = 0.75
val_size = 0.10
test_size = 0.15

valid_exts = ('.jpg', '.png', '.jpeg', '.bmp')

# ---- GET ALL CATEGORY / CLASS COMBINATIONS ----
categories = [d for d in os.listdir(source_dir)
              if os.path.isdir(os.path.join(source_dir, d))]

print("Categories found:", categories)

for category in categories:
    category_path = os.path.join(source_dir, category)

    classes = [d for d in os.listdir(category_path)
               if os.path.isdir(os.path.join(category_path, d))]

    for cls in classes:
        class_path = os.path.join(category_path, cls)

        images = [
            f for f in os.listdir(class_path)
            if f.lower().endswith(valid_exts)
        ]

        if len(images) == 0:
            print(f"Skipping empty class: {category}/{cls}")
            continue

        print(f"\nProcessing {category}/{cls} ({len(images)} images)")

        # ---- SPLIT ----
        train_imgs, temp_imgs = train_test_split(
            images, test_size=(1 - train_size), random_state=42
        )

        val_ratio = val_size / (val_size + test_size)

        val_imgs, test_imgs = train_test_split(
            temp_imgs, test_size=(1 - val_ratio), random_state=42
        )

        splits = {
            'train': train_imgs,
            'val': val_imgs,
            'test': test_imgs
        }

        # ---- COPY FILES ----
        for split_name, split_files in splits.items():
            split_dir = os.path.join(dest_dir, split_name, category, cls)
            os.makedirs(split_dir, exist_ok=True)

            for file in split_files:
                src = os.path.join(class_path, file)
                dst = os.path.join(split_dir, file)

                shutil.copy(src, dst)

print("\n✅ Dataset split complete.")

Categories found: ['Decks', 'Pavements', 'Walls']

Processing Decks/Cracked (2025 images)

Processing Decks/Non-cracked (11595 images)

Processing Pavements/Cracked (2608 images)

Processing Pavements/Non-cracked (21726 images)

Processing Walls/Cracked (3851 images)

Processing Walls/Non-cracked (14287 images)

✅ Dataset split complete.
